## Notebook 4 of 5: Training the classifiers

1. [`01_overview_and_vep.ipynb`](./01_overview_and_vep.ipynb) -- setup, family-aware
   train/test split, gene/ontology descriptions, AlphaGenome reference-only overview plots,
   classical Variant Effect Prediction (VEP).
2. [`02_individual_predictions.ipynb`](./02_individual_predictions.ipynb) -- per-individual
   consensus RNA-seq, comparing two individuals, bulk prediction generation.
3. [`03_population_analysis.ipynb`](./03_population_analysis.ipynb) -- population-level
   comparison, MANE-exon restriction, single-ontology restriction, significance test.
4. [`04_modeling.ipynb`](./04_modeling.ipynb) -- trains a LogisticRegression classifier on
   MANE-exon-restricted features, a second on whole-gene-span features, and a tiny torch MLP
   on the full concatenated signal; persists all three to `notebooks/.cache/models/`.
5. [`05_evaluation.ipynb`](./05_evaluation.ipynb) -- loads the persisted models, evaluates them
   on the held-out test split, reference/variant label prediction, literature-vs-background
   variant comparison.

**This is notebook 4.** It only reads from `notebooks/.cache/predictions/` (notebook 2) and
`notebooks/.cache/analysis/` (notebook 3's persisted `significance_df.csv`/`train_exon_means.csv`)
-- it makes **no AlphaGenome API calls** and needs no `ALPHAGENOME_API_KEY`. It writes
`notebooks/.cache/models/`, which notebook 5 loads instead of re-fitting anything.


In [12]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("/home/breno/I2CA/genomics")
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(NOTEBOOK_DIR)

from utils import setup, rna_features
from alphagenome.models import dna_client

ctx = setup.load_experiment(NOTEBOOK_DIR)
GENES, ONTOLOGY_TERMS, CLASS_MAP = ctx.GENES, ctx.ONTOLOGY_TERMS, ctx.CLASS_MAP
gene_rows, individuals, samples_by_class = ctx.gene_rows, ctx.individuals, ctx.samples_by_class
windows_df, gtf = ctx.windows_df, ctx.gtf
PREDICTIONS_CACHE_DIR = ctx.PREDICTIONS_CACHE_DIR

train_samples_by_class, test_samples_by_class = setup.get_train_test_split(individuals, CLASS_MAP)
for class_name in samples_by_class:
    print(
        f"{class_name}: {len(samples_by_class[class_name])} total -> "
        f"{len(train_samples_by_class[class_name])} train / {len(test_samples_by_class[class_name])} test"
    )

# This notebook only trains models on notebooks/.cache/predictions/ (built by notebook 2) and
# notebooks/.cache/analysis/ (built by notebook 3) -- it makes no AlphaGenome API calls and
# needs no ALPHAGENOME_API_KEY. Notebook 5 (evaluation, reference/variant prediction) does.
ANALYSIS_CACHE_DIR = NOTEBOOK_DIR / ".cache" / "analysis"
MODELS_CACHE_DIR = NOTEBOOK_DIR / ".cache" / "models"
MODELS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Must match notebook 3's choice -- both models below use only this ontology term.
MODEL_ONTOLOGY_TERMS = [ONTOLOGY_TERMS[0]]

gene_zoom_slices = rna_features.compute_gene_zoom_slices(gtf, GENES, dna_client.SEQUENCE_LENGTH_500KB)


strong pigmentation: 703 total -> 570 train / 133 test
weak pigmentation: 369 total -> 292 train / 77 test


## A simple classifier: predicting pigmentation class from RNA-seq

As a first sanity check of how much predictive signal these tracks carry, this trains a plain
`LogisticRegression` on one feature per gene -- the same per-individual, MANE-exon-restricted mean
RNA-seq signal used for the significance test in notebook 3 -- restricted to the genes that came out
`significant (q < 0.05)` there (9 of 11: everything except `OCA2` and `TCHH`, which showed no
detectable difference between classes).

Train features are loaded from `notebooks/.cache/analysis/train_exon_means.csv` (notebook 3's
persisted per-individual scalars, train split only) -- no cache reads here. Test features are
computed fresh here, for `test_samples_by_class` -- the family-aware, population-stratified held-out
individuals that were not involved in picking `significant_genes` -- and persisted below for
notebook 5, which reports accuracy/ROC-AUC on this exact held-out set (genuinely unseen, unrelated
individuals, not the same split the genes were chosen from). Features are standardized (genes
differ hugely in scale, e.g. `DDB1` ~40 vs. `TCHH` ~0.02) and the classifier is class-weight-balanced
(train split is ~570 strong vs. ~292 weak pigmentation individuals). Kept deliberately simple (one
linear model, no tuning/cross-validation) since the goal here is just to see whether these 9 genes'
signal is separable at all, not to produce a tuned predictor.


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

significance_df = pd.read_csv(ANALYSIS_CACHE_DIR / "significance_df.csv")
train_exon_means_df = pd.read_csv(ANALYSIS_CACHE_DIR / "train_exon_means.csv")

significant_genes = significance_df.loc[
    significance_df["significant (q < 0.05)"], "gene"
].tolist()
print(f"Significant genes used as features ({len(significant_genes)}): {significant_genes}")

# Rebuild gene_individual_means (train split only) from the long-format table notebook 3 wrote,
# instead of re-reading the ~47GB prediction cache.
gene_individual_means = {
    gene: {
        class_name: group.set_index("sample_id").loc[sample_ids, "exon_mean"].to_numpy()
        for class_name, sample_ids in train_samples_by_class.items()
    }
    for gene, group in train_exon_means_df.groupby("gene")
}
gene_individual_log_means = {
    gene: {
        class_name: group.set_index("sample_id").loc[sample_ids, "exon_log_mean"].to_numpy()
        for class_name, sample_ids in train_samples_by_class.items()
    }
    for gene, group in train_exon_means_df.groupby("gene")
}

def features_frame(sample_ids_by_class: dict, gene_means: dict) -> pd.DataFrame:
    rows = []
    for class_name, sample_ids in sample_ids_by_class.items():
        for i, sample_id in enumerate(sample_ids):
            row = {"sample_id": sample_id, "label": class_name}
            for gene in significant_genes:
                row[gene] = gene_means[gene][class_name][i]
            rows.append(row)
    return pd.DataFrame(rows)


# Train features: reused from notebook 3's persisted train_exon_means.csv -- no cache reads.
# train_features_df = features_frame(train_samples_by_class, gene_individual_means)
train_features_df = features_frame(train_samples_by_class, gene_individual_log_means)

# Test features: fresh cache reads, restricted to the held-out family-aware split and to just
# the 9 significant genes (not all 11), so this is much cheaper than notebook 3's full pass.
# Persisted below for notebook 5, which evaluates on this exact held-out set without re-reading
# the prediction cache.
t0 = time.time()
test_gene_means = {}
for gene in significant_genes:
    gene_data = windows_df.loc[windows_df["gene"] == gene].iloc[0]
    window_size = gene_data["end"] - gene_data["start"]
    strand = rna_features.gene_strand(gtf, gene)
    mask = rna_features.mane_exon_mask(gtf, gene, gene_data["start"], window_size)
    test_gene_means[gene] = {
        class_name: rna_features.individual_exon_log_means(
            PREDICTIONS_CACHE_DIR, gene, sample_ids, strand, mask, MODEL_ONTOLOGY_TERMS
        )
        for class_name, sample_ids in test_samples_by_class.items()
    }
print(f"Computed test-split features for {len(significant_genes)} genes in {time.time() - t0:.1f}s")

test_features_df = features_frame(test_samples_by_class, test_gene_means)

X_train = train_features_df[significant_genes].values
y_train = (train_features_df["label"] == "strong pigmentation").to_numpy(dtype=int)
X_test = test_features_df[significant_genes].values
y_test = (test_features_df["label"] == "strong pigmentation").to_numpy(dtype=int)

mane_exon_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
mane_exon_model.fit(X_train, y_train)
print("mane_exon_model fit on", X_train.shape[0], "train individuals,", X_train.shape[1], "genes.")


Significant genes used as features (9): ['DDB1', 'HERC2', 'KITLG', 'MC1R', 'MFSD12', 'SLC24A5', 'SLC45A2', 'TYR', 'TYRP1']
Computed test-split features for 9 genes in 86.8s
mane_exon_model fit on 862 train individuals, 9 genes.


### Follow-up: whole-gene span instead of MANE-exon-restricted signal

The classifier above collapsed each significant gene down to `log(mean + 0.001)` of the RNA-seq
signal restricted to the gene's **MANE Select transcript exons only** (introns and flanking
sequence excluded via `mane_exon_mask`). This repeats the exact same one-scalar-per-gene,
log-transformed setup, but takes the mean over the gene's **entire GTF span** instead
(`gene_zoom_slices`, already computed in the setup cell -- every transcript/isoform, not just
the MANE Select transcript's exons) -- to check whether restricting to exons was actually helping,
or whether intronic/flanking signal carries comparable predictive information.

Still uses the same `significant_genes` selected by notebook 3's exon-based significance test (no
separate whole-gene significance test was run) and the same family-aware train/test split. Unlike
the cell above, neither split can reuse a persisted CSV here -- notebook 3 only wrote
exon-restricted means -- so both train and test features are computed fresh from
`PREDICTIONS_CACHE_DIR`.


In [14]:
t0 = time.time()
gene_span_means_by_split = {}
for split_name, sample_ids_by_class in [("train", train_samples_by_class), ("test", test_samples_by_class)]:
    gene_span_means_by_split[split_name] = {}
    for gene in significant_genes:
        strand = rna_features.gene_strand(gtf, gene)
        lo, hi = gene_zoom_slices[gene]
        gene_span_means_by_split[split_name][gene] = {
            class_name: rna_features.individual_gene_log_means(
                PREDICTIONS_CACHE_DIR, gene, sample_ids, strand, lo, hi, MODEL_ONTOLOGY_TERMS
            )
            for class_name, sample_ids in sample_ids_by_class.items()
        }
print(f"Computed whole-gene-span features for {len(significant_genes)} genes, both splits, in {time.time() - t0:.1f}s")

# Persisted below for notebook 5 -- same reasoning as test_features_df above.
train_gene_features_df = features_frame(train_samples_by_class, gene_span_means_by_split["train"])
test_gene_features_df = features_frame(test_samples_by_class, gene_span_means_by_split["test"])

X_train_gene = train_gene_features_df[significant_genes].values
y_train_gene = (train_gene_features_df["label"] == "strong pigmentation").to_numpy(dtype=int)
X_test_gene = test_gene_features_df[significant_genes].values
y_test_gene = (test_gene_features_df["label"] == "strong pigmentation").to_numpy(dtype=int)

gene_span_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
gene_span_model.fit(X_train_gene, y_train_gene)
print("gene_span_model fit on", X_train_gene.shape[0], "train individuals,", X_train_gene.shape[1], "genes.")


Computed whole-gene-span features for 9 genes, both splits, in 379.5s
gene_span_model fit on 862 train individuals, 9 genes.


## A tiny torch model: full gene-span signal, all genes, concatenated

The classifier above collapsed each gene down to a single scalar (its exon-only mean) and used
only the 9 genes that passed the significance test. This instead feeds a small neural network
the much richer **raw per-position signal**, for **every gene** (not just the significant ones),
concatenated into one long vector per individual:

- Per gene: combined (H1+H2, restricted to `MODEL_ONTOLOGY_TERMS`) signal, restricted to the
  gene's own transcribed strand, cropped to the same window notebook 2's
  `plot_rna_seq_comparison` / notebook 3's `gene_zoom_slices` already use -- the whole GTF
  **gene** feature span (every transcript/isoform, not just the MANE Select transcript's exons),
  recomputed above via `rna_features.compute_gene_zoom_slices`.
- All 11 genes' cropped signals are concatenated end-to-end into one vector per individual.

This is a genuinely different, much higher-dimensional regime than the scalar classifier: the
concatenated vector is roughly 950k values long, against ~860 train individuals -- an extreme
`p >> n` setting, so expect heavy overfitting risk even with a tiny model; treat this as a rough
sanity check (can *any* small net extract more than the scalar-mean features did?), not a tuned
result. Dropout + weight decay provide some regularization, and the same family-aware,
population-stratified train/test split from notebook 1 is reused so the comparison to the earlier
classifier is apples-to-apples.

Building the feature matrices below re-reads the full prediction cache **for every individual in
both splits** (unlike the significance test in notebook 3, which only needed the train split, this
also needs raw per-position test-split arrays that were never kept anywhere) -- expect a similar
multi-minute read as notebook 3's very first population-mean pass. Building the two dense float32
matrices needs a few GB of RAM. Training itself uses the GPU if available
(`torch.cuda.is_available()`), since a ~950k-dimensional input makes even one linear layer
expensive on CPU.


In [15]:
gene_widths = {gene: hi - lo for gene, (lo, hi) in gene_zoom_slices.items()}
gene_offsets = {}
_offset = 0
for gene in GENES:
    gene_offsets[gene] = _offset
    _offset += gene_widths[gene]
feature_dim = _offset
print(f"Concatenated feature dimension: {feature_dim:,}")
print(", ".join(f"{gene}={gene_widths[gene]:,}" for gene in GENES))


def build_feature_matrix(sample_ids_by_class: dict):
    sample_ids, labels = [], []
    for class_name, sids in sample_ids_by_class.items():
        sample_ids.extend(sids)
        labels.extend([1.0 if class_name == "strong pigmentation" else 0.0] * len(sids))
    labels = np.array(labels, dtype=np.float32)

    X = np.empty((len(sample_ids), feature_dim), dtype=np.float32)
    t0 = time.time()
    for gene in GENES:
        strand = rna_features.gene_strand(gtf, gene)
        lo, hi = gene_zoom_slices[gene]
        offset = gene_offsets[gene]
        for row, sample_id in enumerate(sample_ids):
            X[row, offset:offset + gene_widths[gene]] = rna_features.individual_gene_span_signal(
                PREDICTIONS_CACHE_DIR, gene, sample_id, strand, lo, hi, MODEL_ONTOLOGY_TERMS
            )
        print(f"[{time.time() - t0:5.1f}s] {gene}: done ({len(sample_ids)} individuals)")
    return X, labels, sample_ids


print("Building train feature matrix...")
X_train_raw, y_train_nn, nn_train_ids = build_feature_matrix(train_samples_by_class)
print("\nBuilding test feature matrix...")
X_test_raw, y_test_nn, nn_test_ids = build_feature_matrix(test_samples_by_class)

nn_scaler = StandardScaler()
X_train_nn = nn_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_nn = nn_scaler.transform(X_test_raw).astype(np.float32)
del X_train_raw, X_test_raw
print(f"\nX_train_nn: {X_train_nn.shape}, X_test_nn: {X_test_nn.shape}")


Concatenated feature dimension: 1,010,926
MC1R=8,854, TYRP1=24,846, TYR=117,884, SLC45A2=40,070, MFSD12=36,030, OCA2=344,440, HERC2=211,140, SLC24A5=21,682, DDB1=43,146, KITLG=88,058, ASIP=74,776
Building train feature matrix...
[ 36.4s] MC1R: done (862 individuals)
[ 68.8s] TYRP1: done (862 individuals)
[101.5s] TYR: done (862 individuals)
[135.5s] SLC45A2: done (862 individuals)
[171.6s] MFSD12: done (862 individuals)
[206.9s] OCA2: done (862 individuals)
[240.6s] HERC2: done (862 individuals)
[272.5s] SLC24A5: done (862 individuals)
[306.7s] DDB1: done (862 individuals)
[338.2s] KITLG: done (862 individuals)
[373.2s] ASIP: done (862 individuals)

Building test feature matrix...
[  9.3s] MC1R: done (210 individuals)
[ 18.0s] TYRP1: done (210 individuals)
[ 26.6s] TYR: done (210 individuals)
[ 35.4s] SLC45A2: done (210 individuals)
[ 44.8s] MFSD12: done (210 individuals)
[ 53.9s] OCA2: done (210 individuals)
[ 62.7s] HERC2: done (210 individuals)
[ 71.1s] SLC24A5: done (210 individual

In [16]:
import torch
import torch.nn as nn

from utils.models import TinyMLP

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

X_train_t = torch.from_numpy(X_train_nn).to(device)
y_train_t = torch.from_numpy(y_train_nn).to(device)
X_test_t = torch.from_numpy(X_test_nn).to(device)
y_test_t = torch.from_numpy(y_test_nn).to(device)

torch.manual_seed(0)
nn_model = TinyMLP(feature_dim).to(device)
print(f"Model parameters: {sum(p.numel() for p in nn_model.parameters()):,}")

# Class-imbalance correction (~570 strong vs. ~292 weak in train), same spirit as
# class_weight="balanced" in the LogisticRegression cells above.
pos_weight = torch.tensor([(y_train_nn == 0).sum() / (y_train_nn == 1).sum()], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3, weight_decay=1e-3)

# train_acc/test_acc below are a live training-time diagnostic (are we overfitting yet?), not
# a final evaluation -- the held-out metrics report is notebook 5's job, on the persisted
# nn_model + persisted test features, not recomputed here.
N_EPOCHS = 50
for epoch in range(N_EPOCHS):
    nn_model.train()
    optimizer.zero_grad()
    loss = criterion(nn_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0 or epoch == N_EPOCHS - 1:
        nn_model.eval()
        with torch.no_grad():
            train_acc = ((torch.sigmoid(nn_model(X_train_t)) > 0.5).float() == y_train_t).float().mean().item()
            test_acc = ((torch.sigmoid(nn_model(X_test_t)) > 0.5).float() == y_test_t).float().mean().item()
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}  train_acc {train_acc:.3f}  test_acc {test_acc:.3f}")

nn_model.eval()
print("nn_model training complete.")


Using device: cuda
Model parameters: 32,349,697
epoch   0  loss 0.4666  train_acc 0.836  test_acc 0.829
epoch   5  loss 2.4344  train_acc 0.987  test_acc 0.976
epoch  10  loss 0.7062  train_acc 0.997  test_acc 0.990
epoch  15  loss 0.4744  train_acc 0.998  test_acc 0.995
epoch  20  loss 0.4907  train_acc 1.000  test_acc 0.995
epoch  25  loss 0.0019  train_acc 1.000  test_acc 0.990
epoch  30  loss 0.0654  train_acc 1.000  test_acc 0.990
epoch  35  loss 0.6001  train_acc 1.000  test_acc 0.995
epoch  40  loss 0.0252  train_acc 1.000  test_acc 0.990
epoch  45  loss 0.0023  train_acc 1.000  test_acc 0.990
epoch  49  loss 0.0004  train_acc 1.000  test_acc 0.990
nn_model training complete.


## Persisting the trained models for notebook 5

All evaluation (held-out accuracy/ROC-AUC/confusion matrices, coefficient/logit inspection,
the reference-genome/variant label predictions, and the literature-vs-background rain-plot
comparison) moved to **[`05_evaluation.ipynb`](./05_evaluation.ipynb)**, so it can be re-run
independently without re-fitting anything here. This persists everything that notebook needs,
under `notebooks/.cache/models/`:

- `mane_exon_model.joblib` / `gene_span_model.joblib` -- the two `StandardScaler` +
  `LogisticRegression` pipelines (`joblib`, sklearn's recommended format).
- `nn_model_state_dict.pt` -- the tiny MLP's weights only (`torch.save(model.state_dict())`,
  the recommended-over-pickling-the-whole-module approach) -- notebook 5 rebuilds the
  architecture from `utils.models.TinyMLP` (the same class this notebook trained) before
  loading these weights in.
- `nn_scaler.joblib` -- the `StandardScaler` fit on the concatenated per-position features,
  needed to scale any new input the same way before feeding `nn_model`.
- `test_features_mane_exon.csv` / `test_features_gene_span.csv` -- the two scalar-feature test
  sets (`test_features_df`/`test_gene_features_df`), so notebook 5 evaluates on the exact same
  held-out individuals without re-reading `notebooks/.cache/predictions/`.
- `test_features_nn.npz` -- `X_test_nn`/`y_test_nn`/`nn_test_ids`, the ~800MB dense concatenated
  test matrix -- by far the most expensive artifact to rebuild (the multi-minute full-cache
  read in the cell above), so persisting it is what makes notebook 5 cheap to re-run.
- `model_metadata.json` -- `significant_genes`, `GENES` (concatenation order), `gene_widths`,
  `gene_offsets`, `feature_dim`, `MODEL_ONTOLOGY_TERMS`, and the tiny MLP's `hidden_dim` --
  everything notebook 5 needs to reconstruct feature vectors/model architecture consistently
  with how they were built here, without re-deriving them from `gtf`/`gene_zoom_slices`.


In [17]:
import joblib

from utils import models as models_utils

models_utils.save_sklearn_model(mane_exon_model, MODELS_CACHE_DIR / "mane_exon_model.joblib")
models_utils.save_sklearn_model(gene_span_model, MODELS_CACHE_DIR / "gene_span_model.joblib")
models_utils.save_nn_model(nn_model, MODELS_CACHE_DIR / "nn_model_state_dict.pt")
joblib.dump(nn_scaler, MODELS_CACHE_DIR / "nn_scaler.joblib")

test_features_df.to_csv(MODELS_CACHE_DIR / "test_features_mane_exon.csv", index=False)
test_gene_features_df.to_csv(MODELS_CACHE_DIR / "test_features_gene_span.csv", index=False)
np.savez(
    MODELS_CACHE_DIR / "test_features_nn.npz",
    X=X_test_nn, y=y_test_nn, sample_ids=np.array(nn_test_ids),
)

models_utils.save_metadata(
    {
        "significant_genes": significant_genes,
        "GENES": GENES,
        "gene_widths": gene_widths,
        "gene_offsets": gene_offsets,
        "feature_dim": feature_dim,
        "MODEL_ONTOLOGY_TERMS": MODEL_ONTOLOGY_TERMS,
        "nn_hidden_dim": 32,
    },
    MODELS_CACHE_DIR / "model_metadata.json",
)

saved = sorted(p.name for p in MODELS_CACHE_DIR.iterdir())
print(f"Saved {len(saved)} artifact(s) to {MODELS_CACHE_DIR}:")
for name in saved:
    size_mb = (MODELS_CACHE_DIR / name).stat().st_size / 1e6
    print(f"  {name:<32} {size_mb:8.1f} MB")


Saved 8 artifact(s) to /home/breno/I2CA/genomics/notebooks/.cache/models:
  gene_span_model.joblib                0.0 MB
  mane_exon_model.joblib                0.0 MB
  model_metadata.json                   0.0 MB
  nn_model_state_dict.pt              129.4 MB
  nn_scaler.joblib                     24.3 MB
  test_features_gene_span.csv           0.0 MB
  test_features_mane_exon.csv           0.0 MB
  test_features_nn.npz                849.2 MB
